In [8]:
import os
import re
import requests
import json
import nest_asyncio
from pathlib import Path
from datetime import datetime
from tqdm.notebook import tqdm
from tika import parser
import pandas as pd

In [9]:
from rag.lib.metadata import build_metadata
from rag.lib.config import LEXES_API_URL
from rag.lib.downloads import download_documents

In [10]:
DATA_DIR = Path("data")
STATS_DIR = Path("stats")

In [11]:
nest_asyncio.apply()

response = requests.get(LEXES_API_URL)
if response.status_code != 200:
    # TODO : a gerer dans les logs
    raise Exception(f"Unexpected status code while fetching : {response.status_code}")

# TODO : remove debugging param
data = build_metadata(response, debugging=True)

download_documents(data, DATA_DIR)

No SPARQL results for https://fedlex.data.admin.ch/redirectInfo?url=eli%2Fcc%2F1993%2F210_210_210%2Ffr&original=https:%2F%2Fwww.admin.ch%2Fch%2Ff%2Frs%2Fc414_110.html


In [12]:
# filter pdf to index based on heuristic (np_pages + nb_occ_article) and hard coded values

ARTICLE_PATTERN = re.compile(r"\b(?:Article\s+\d+|Art\.\s*\d+)\b")

stats_per_doc = []

for file in tqdm(DATA_DIR.glob("*.pdf")):
    parsed_file = parser.from_file(str(file))
    metadata = parsed_file.get("metadata", {})
    nb_pages = int(metadata.get("xmpTPg:NPages"))
    extracted_text = parsed_file['content']
    nb_occ_article = sum(1 for _ in ARTICLE_PATTERN.finditer(extracted_text))
    stats_per_doc.append({
        "doc_id": file.stem,
        "nb_pages": nb_pages,
        "nb_occ_article": nb_occ_article
    })

stats = pd.DataFrame(stats_per_doc)
pdfs_not_to_index = stats.loc[(stats["nb_pages"] > 100) | ((stats["nb_pages"] > 50) & (stats["nb_occ_article"] == 0))]["doc_id"].values
print(pdfs_not_to_index)

0it [00:00, ?it/s]

<ArrowStringArray>
[]
Length: 0, dtype: str


In [13]:
# add key 'is_indexed' in metadata dict

for content, metadata_dict in data.items():
    metadata = metadata_dict
    doc_id = metadata["doc_id"]
    if doc_id in pdfs_not_to_index:
        metadata["is_indexed"] = False
    else:
        metadata["is_indexed"] = True
    data[content] = metadata

In [14]:
# load actual state of metadata in json file

os.makedirs(STATS_DIR, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
metadata_filename = os.path.join(STATS_DIR, f"{timestamp}_lexes_metadata.json")

with open(metadata_filename, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=4, ensure_ascii=False)